# Donor-bulk production quality control

Data-only QC report for donor raw-count aggregation, matched controls, CLAMPfull models, grouped folds, all-cell projection, and independent expression UMAPs.


💡 **Environment:** `clamp-analyses`


In [ ]:
from pathlib import Path
import pandas as pd

DATASETS = list(snakemake.params.datasets)
OUT = Path(snakemake.params.out_dir)
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
def read_many(paths, source):
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        if "dataset" not in frame.columns:
            parts = Path(path).parts
            frame["dataset"] = parts[parts.index("donor_bulk") + 1]
        frame["qc_source"] = source
        frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False)

aggregation = read_many(list(snakemake.input.aggregation), "aggregation")
ranks = read_many(list(snakemake.input.ranks), "rank")
projections = read_many(list(snakemake.input.projections), "projection")
umaps = read_many(list(snakemake.input.umaps), "expression_umap")
validation = pd.read_csv(snakemake.input.validation)

stopifnot = lambda condition, message: condition or (_ for _ in ()).throw(AssertionError(message))
stopifnot(set(aggregation.dataset) == set(DATASETS), "aggregation coverage")
stopifnot(set(projections.dataset) == set(DATASETS), "projection coverage")
stopifnot(set(umaps.dataset) == set(DATASETS), "UMAP coverage")
stopifnot(validation.shape[0] == len(DATASETS), "validation coverage")
stopifnot(validation.truth_denominator_exact.all(), "truth denominators")
stopifnot(validation.folds_leakage_free.all(), "donor leakage")
stopifnot(validation.all_model_matrices_finite.all(), "finite models")
stopifnot(validation.expression_umap_independent.all(), "expression UMAP source")


In [ ]:
rank_summary = ranks.groupby("dataset", as_index=False)["k"].first().rename(columns={"k": "selected_rank"})
qc = (validation
      .merge(aggregation[["dataset", "n_cells_raw", "n_cells_retained", "n_cells_sampled",
                          "n_genes_raw", "n_genes_output", "raw_validation_scope",
                          "truth_denominators_equal_expression_cells"]], on="dataset", validate="one_to_one")
      .merge(projections[["dataset", "n_cells_projected", "n_cells_mapped",
                          "duplicate_gene_policy", "normalization"]], on="dataset", validate="one_to_one")
      .merge(umaps[["dataset", "n_cells", "n_hvg", "n_pcs",
                    "embedding_source", "seed"]], on="dataset", validate="one_to_one")
      .merge(rank_summary, on="dataset", validate="one_to_one"))
qc.to_csv(snakemake.output.summary, index=False)
qc


The source notebook is deliberately output-clean. Snakemake stores this table and the executed notebook; no figure files are produced here.
